# Week 6 — Agentic RAG (LangGraph)

5주차 우승 retriever(Dense) 위에 `retrieve→grade→rewrite→generate`(+retry, +reflect) 구조.

**통제 변수:** 데이터 5건 / B-large 1500·300 / bge-m3 / gpt-4o-mini(temp=0) / 10문항 / RAGAS 4지표. Baseline·V0·V1 모두 `study_agentic_clean` collection 공통 — 바뀌는 건 routing 구조뿐.

**5주차 실패 3케이스:** A) 도입부 노이즈 top-k 잠식(Q10) / B) 다른 영상 의미 표류(Q2) / C) broad query 발산(Q1).

## 0) Preprocessing (LLM utility 분류)
도입부/꼬리 노이즈를 chunk-level로 분류 시도. **결과: 1500자 granularity에선 거의 0개 drop**(노이즈가 큰 chunk에 희석). negative finding.

In [1]:
from advanced_retrieval.chunks import load_chunks
from agentic_rag.preprocess import ingest_clean
raw = len(load_chunks()); kept = ingest_clean()
print(f'raw={raw}  kept(clean)={kept}  dropped={raw-kept}')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

raw=68  kept(clean)=68  dropped=0


## 1) 그래프 빌드 (V0 base / V1 +reflect)

In [2]:
from agentic_rag.graph import build_graph
from agentic_rag.state import initial_state
g0 = build_graph(with_reflect=False)
g1 = build_graph(with_reflect=True)
print('built V0, V1')

/Users/soom/Projects/investment-thesis-lens-rag/.venv/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

built V0, V1


## 2) 단일 질문 trace (라우팅 확인)

In [3]:
f = g0.invoke(initial_state('GTC 워싱턴에서 엔비디아가 발표한 신규 사업 영역은?'))
print('route:', ' > '.join(f['route_history']), '| retry:', f['retry_count'])
print(f['answer'][:200])

route: retrieve > grade > generate | retry: 0
주어진 자료로는 알 수 없습니다. 

[출처]
- 2025-11-03 젠슨황의 깐부회동과 트럼프 순방이 끌어올린 증시, 주목받을 수혜주들


## 3) 정량 비교 — Baseline / V0 / V1
동일 10문항. Baseline=clean collection + plain chain, V0/V1=agentic graph. RAGAS 4 + 평균 latency + 평균 LLM 호출수.

In [4]:
import pandas as pd
from agentic_rag.eval import eval_graph, eval_chain
from agentic_rag.retrieval import build_clean_retriever

base_df = eval_chain(build_clean_retriever())
v0_df = eval_graph(g0)
v1_df = eval_graph(g1)
table = pd.DataFrame({
    'Baseline': base_df.mean(numeric_only=True),
    'V0': v0_df.mean(numeric_only=True),
    'V1': v1_df.mean(numeric_only=True),
}).T.round(4)
table.to_csv('week6_results.csv'); table

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,latency_s,llm_calls
Baseline,0.9378,0.6253,0.9556,0.9250,4.1809,1.0
V0,1.0000,0.6265,0.9932,0.9667,8.4862,2.0
V1,0.8532,0.6305,0.9932,0.9667,8.7829,3.0


## 4) Preprocessing 효과 분리 — 원본 vs clean baseline
원본 collection(study_advanced_retrieval) baseline vs clean baseline. clean==원본(0 drop)이라 차이 거의 없을 것으로 예상 — negative finding 확인.

In [5]:
from advanced_retrieval.retrievers import build_dense as build_orig_dense
orig_df = eval_chain(build_orig_dense())
prep = pd.DataFrame({
    '원본 baseline': orig_df.mean(numeric_only=True),
    'clean baseline': base_df.mean(numeric_only=True),
}).T.round(4)
prep.to_csv('week6_prep_effect.csv'); prep

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,latency_s,llm_calls
원본 baseline,0.9689,0.6280,0.9556,0.925,3.8430,1.0
clean baseline,0.9378,0.6253,0.9556,0.925,4.1809,1.0


## 5) 실패 3케이스 재검토 (C=Q1 broad / B=Q2 표류 / A=Q10 도입부)
V0/V1으로 다시 돌려 route_history + 답변 확인 → 구조로 해결/부분/여전히 실패 판정.

In [6]:
from naive_rag.constants import WEEK4_EVAL_DATA
cases = {'C(Q1 broad)': 0, 'B(Q2 표류)': 1, 'A(Q10 도입부)': 9}
lines = []
for label, i in cases.items():
    q = WEEK4_EVAL_DATA[i]['question']
    block = [f'### {label}: {q}']
    for name, g in [('V0', g0), ('V1', g1)]:
        f = g.invoke(initial_state(q))
        block.append(f"  [{name}] route: {' > '.join(f['route_history'])} | retry {f['retry_count']}")
        block.append(f"        answer: {f['answer'][:160]}")
    lines.append(chr(10).join(block))
dump = (chr(10)*2).join(lines)
open('week6_failure_review.txt','w').write(dump)
print(dump)

### C(Q1 broad): 최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?
  [V0] route: retrieve > grade > generate | retry 0
        answer: 최근 AI 투자 전쟁에서 주목한 수혜주는 구글과 아마존입니다. 이 두 기업은 엔트로픽에 대규모 투자를 하였으며, 특히 구글은 기존 투자 대비 10배 이상의 규모로 추가 투자한다고 밝혔습니다. 아마존도 최소 3배 이상의 금액을 대규모로 추가 투자할 계획입니다. 이러한 투자로 인해 두 기업
  [V1] route: retrieve > grade > generate > reflect(ok) | retry 0
        answer: 최근 AI 투자 전쟁에서 주목한 수혜주는 구글과 아마존입니다. 이 두 기업은 엔트로픽에 대규모 투자를 하였으며, 특히 구글은 기존 투자 대비 10배 이상의 규모로 추가 투자한다고 밝혔습니다. 아마존도 최소 3배 이상의 금액을 대규모로 추가 투자할 계획입니다. 이 외에도 엔비디아와 ARM

### B(Q2 표류): AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?
  [V0] route: retrieve > grade > generate | retry 0
        answer: AI 에이전트의 확산은 반도체 주도주 판도를 크게 변화시키고 있습니다. 기존에는 엔비디아의 GPU가 AI 붐을 주도했으나, 이제는 CPU의 중요성이 급격히 증가하고 있습니다. AI 에이전트 시대에서는 CPU가 실무형 직원으로서 여러 업무를 동시에 처리해야 하므로, CPU의 수요가 증가하
  [V1] route: retrieve > grade > generate > reflect(ok) | retry 0
        answer: AI 에이전트의 확산은 반도체 주도주 판도를 크게 변화시키고 있습니다. 기존에는 엔비디아의 GPU가 AI 붐을 주도했으나, 이제는 CPU의 중요성이 급격히 증가하고 있습니다. AI 에이전트 시대에서는 CPU가 실무형 직원

## 6) 운영지표 — 질문별 latency / LLM 호출수 / route

In [7]:
ops = v0_df[['latency_s','llm_calls','route_history']].copy()
ops.insert(0, 'question', [d['question'][:24] for d in WEEK4_EVAL_DATA])
ops.to_csv('week6_ops.csv', index=False)
print('Baseline 평균 latency:', round(base_df['latency_s'].mean(),3),
      '| V0:', round(v0_df['latency_s'].mean(),3),
      '| V1:', round(v1_df['latency_s'].mean(),3))
print('Baseline 평균 LLM호출:', round(base_df['llm_calls'].mean(),2),
      '| V0:', round(v0_df['llm_calls'].mean(),2),
      '| V1:', round(v1_df['llm_calls'].mean(),2))
ops

Baseline 평균 latency: 4.181 | V0: 8.486 | V1: 8.783
Baseline 평균 LLM호출: 1.0 | V0: 2.0 | V1: 3.0


,question,latency_s,llm_calls,route_history
0,최근 AI 투자 전쟁에서 주목한 수혜주는 무,23.219590,2,retrieve > grade > generate
1,AI 에이전트가 반도체 주도주 판도를 어떻게,10.637382,2,retrieve > grade > generate
2,트럼프 순방과 젠슨황의 깐부회동이 증시에 미,7.484768,2,retrieve > grade > generate
3,2026 버블 시나리오에서 본격 버블은 시작,5.018299,2,retrieve > grade > generate
4,최근 폭락한 미국 초우량주 중 매수 후보로,5.528278,2,retrieve > grade > generate
5,"헬스케어 섹터(유나이티드헬스, 노보노 등)에",7.784594,2,retrieve > grade > generate
6,"인텔, AMD, ARM 등 반도체 종목에 대",6.347115,2,retrieve > grade > generate
7,GTC 워싱턴 행사에서 엔비디아가 발표한 신,5.145617,2,retrieve > grade > generate
8,2030년까지 컴퓨팅 용량 확장과 AI 인프,6.424470,2,retrieve > grade > generate
9,"빅테크(아마존, 구글, 엔비디아) 중 AI",7.272340,2,retrieve > grade > generate


## 7) 회고 요약

| 구성 | Faithful | AnswerRel | CtxPrecision | CtxRecall | latency(s) | LLM호출 |
|---|---|---|---|---|---|---|
| Baseline | 0.938 | 0.625 | 0.956 | 0.925 | 4.2 | 1.0 |
| **V0** | **1.000** | 0.627 | **0.993** | **0.967** | 8.5 | 2.0 |
| V1(+reflect) | 0.853 | 0.631 | 0.993 | 0.967 | 8.8 | 3.0 |

- **V0 채택**: grade 필터링(precision↑) + neighbor expansion(recall↑)으로 baseline 대비 개선. 비용 LLM 1콜·latency 2배.
- **rewrite/retry 루프는 10문항 전부 발화 안 함**(grade가 항상 enough). 실패 3케이스를 푼 건 비싼 루프가 아니라 싼 grade 필터링.
- **reflect(V1)은 Faithfulness 1.0→0.853으로 악화** + LLM +50% → 기각.
- **Preprocessing 효과 0**: 1500자 chunk-level 분류가 0 drop(노이즈는 문장 단위). 원본≈clean.

전체 분석: [`docs/week6_retrospective.md`](../docs/week6_retrospective.md) · [`docs/adr/week6_agentic_rag.md`](../docs/adr/week6_agentic_rag.md)